# Yoga Pose Classification (Image-based)

This notebook demonstrates training an EfficientNetB0-based yoga pose classifier on the **Yoga-82** dataset.

**Requirements:** Yoga-82 dataset must be placed in `data/raw/yoga82/` and preprocessed via `scripts/preprocess.py --yoga82`

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))
print('Root:', ROOT)

In [ ]:
# Check dataset availability
from scripts.validate_datasets import CHECKS
for name, info in CHECKS.items():
    status = '✅' if info['path'].exists() else '❌'
    print(f'{status} {name}')

In [ ]:
# Check processed data
processed = ROOT / 'data' / 'processed' / 'yoga82'
for split in ['train', 'val', 'test']:
    p = processed / split
    if p.exists():
        n_cls = sum(1 for x in p.iterdir() if x.is_dir())
        n_imgs = sum(1 for x in p.rglob('*.jpg'))
        print(f'{split}: {n_cls} classes, {n_imgs} images')
    else:
        print(f'{split}: NOT FOUND — run scripts/preprocess.py --yoga82')

In [ ]:
# Build and summarise model
from ml.models.yoga_classifier import build_yoga_classifier

meta_path = ROOT / 'data' / 'processed' / 'metadata' / 'class_to_idx.json'
if meta_path.exists():
    with open(meta_path) as f:
        class_to_idx = json.load(f)
    num_classes = len(class_to_idx)
    print(f'Number of classes: {num_classes}')
    model = build_yoga_classifier(num_classes=num_classes, fine_tune=False)
    model.summary()
else:
    print('class_to_idx.json not found — run preprocessing first')

In [ ]:
# Training (Phase 1 — head only)
# Uncomment and run after placing data
#
# !python -m ml.training.train_yoga_classifier --subset 50 --epochs-p1 5 --epochs-p2 5
# (--subset 50 uses 50 images/class for quick dev testing)
print('To train: python -m ml.training.train_yoga_classifier --subset 50')

In [ ]:
# Load and inspect metrics (after training)
metrics_path = ROOT / 'models' / 'metrics' / 'yoga_classifier_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print(f"Accuracy : {m['accuracy']}")
    print(f"Precision: {m['precision']}")
    print(f"Recall   : {m['recall']}")
    print(f"F1 Score : {m['f1_score']}")
    print(f"Data src : {m['data_source']}")
    print(f"N samples: {m['n_val_samples']}")
else:
    print('No metrics found — train the model first')

In [ ]:
# Confusion matrix visualisation (after training)
import json
metrics_path = ROOT / 'models' / 'metrics' / 'yoga_classifier_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    import numpy as np
    import matplotlib.pyplot as plt
    cm = np.array(m['confusion_matrix'])
    plt.figure(figsize=(14, 12))
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    plt.title(f"Yoga-82 Confusion Matrix (accuracy={m['accuracy']})")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()
else:
    print('Train the model first to see the confusion matrix')

In [ ]:
# Single-image inference (after training)
# Replace with a real image path from the test set
# from ml.inference.predictor import classify_yoga_image
# result = classify_yoga_image('data/processed/yoga82/test/Warrior_II/img001.jpg')
# print(result)
print('Uncomment above after training')

## Limitations
- Metrics above are on the **validation set**, not a held-out test set used for paper claims.
- Real-world performance will vary with image quality, background, and pose variation.
- This is an academic prototype — no clinical validity.